# EDA Caltech101 & CUB-200-2011 for Image-to-Image Retrieval

Notebook này chạy trực tiếp trên Kaggle và tạo EDA nhanh cho Caltech101 và CUB-200-2011 để hỗ trợ quyết định pipeline image-to-image retrieval.

Phạm vi:
- Tự tải dữ liệu từ CaltechDATA nếu chưa có trong `/kaggle/working/datasets`.
- Đọc metadata, kích thước ảnh, split, class, và bounding box của CUB.
- Tạo CSV reports trong `/kaggle/working/eda_outputs/reports`.
- Tạo PNG figures trong `/kaggle/working/eda_outputs/figures`.
- Chạy baseline retrieval bằng CLIP image encoder từ `open_clip_torch`.
- Cache embedding trong `/kaggle/working/eda_outputs/cache`.

Nguồn dữ liệu chính thức:
- Caltech101: https://data.caltech.edu/records/mzrjq-6wc02
- CUB-200-2011: https://data.caltech.edu/records/65de6-vp158

## 1. Cài đặt package nếu thiếu

Kaggle thường đã có PyTorch, pandas, numpy, PIL, matplotlib và tqdm. Cell này chỉ cài `open_clip_torch` nếu môi trường chưa có.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "open_clip": "open_clip_torch",
}

missing = [pip_name for import_name, pip_name in required.items() if importlib.util.find_spec(import_name) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All optional packages are already available.")

## 2. Import thư viện

In [ ]:
import hashlib
import math
import os
import random
import tarfile
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from IPython.display import display
from PIL import Image, ImageDraw, ImageFile
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

import open_clip

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))

## 3. Config đường dẫn và tham số

Tất cả dữ liệu và output được đặt dưới `/kaggle/working` để notebook chạy đúng trên Kaggle.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_DIR = Path("/kaggle/working")
DATA_DIR = BASE_DIR / "datasets"
OUT_DIR = BASE_DIR / "eda_outputs"
REPORT_DIR = OUT_DIR / "reports"
FIG_DIR = OUT_DIR / "figures"
CACHE_DIR = OUT_DIR / "cache"

for d in [DATA_DIR, OUT_DIR, REPORT_DIR, FIG_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CALTECH101_URL = "https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip?download=1"
CUB_URL = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1"

ARCHIVES = {
    "caltech101": {
        "url": CALTECH101_URL,
        "path": DATA_DIR / "caltech-101.zip",
        "md5": "3138e1922a9193bfa496528edbbc45d0",
    },
    "cub2002011": {
        "url": CUB_URL,
        "path": DATA_DIR / "CUB_200_2011.tgz",
        "md5": "97eceeb196236b17998738112f37df78",
    },
}

MODEL_NAME = "ViT-B-32"
PRETRAINED = "openai"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 64 if DEVICE == "cuda" else 16
NUM_WORKERS = 2
TOPK = 10
SIM_CHUNK_SIZE = 256 if DEVICE == "cuda" else 64
PAIR_SIM_SAMPLES = 50000
SHOW_FIGURES = True

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("DEVICE:", DEVICE)

## 4. Download và extract dataset

Cell này tự tải dữ liệu nếu archive chưa tồn tại hoặc checksum không khớp. Nếu dataset đã được extract, notebook sẽ bỏ qua bước extract.

Lưu ý Kaggle: nếu download lỗi do network, hãy bật Internet trong Notebook Settings.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def file_md5(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def download_file(url, dst, expected_md5=None, retries=3):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        if expected_md5 is None:
            print(f"Archive exists, skip download: {dst}")
            return dst
        current = file_md5(dst)
        if current == expected_md5:
            print(f"Archive exists and MD5 matches, skip download: {dst.name}")
            return dst
        print(f"MD5 mismatch for {dst.name}: {current} != {expected_md5}. Re-downloading.")
        dst.unlink()

    tmp = dst.with_suffix(dst.suffix + ".part")
    if tmp.exists():
        tmp.unlink()

    last_error = None
    headers = {"User-Agent": "Mozilla/5.0"}
    for attempt in range(1, retries + 1):
        try:
            print(f"Downloading {dst.name} (attempt {attempt}/{retries})")
            with requests.get(url, stream=True, allow_redirects=True, timeout=(30, 120), headers=headers) as r:
                r.raise_for_status()
                total = int(r.headers.get("content-length", 0))
                with open(tmp, "wb") as f, tqdm(total=total if total > 0 else None, unit="B", unit_scale=True, desc=dst.name) as pbar:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            tmp.rename(dst)
            if expected_md5 is not None:
                current = file_md5(dst)
                if current != expected_md5:
                    raise RuntimeError(f"MD5 mismatch after download: {current} != {expected_md5}")
            print(f"Downloaded: {dst}")
            return dst
        except Exception as exc:
            last_error = exc
            print(f"Download failed: {exc}")
            if tmp.exists():
                tmp.unlink()
            time.sleep(5 * attempt)

    raise RuntimeError(
        "Download failed after retries. On Kaggle, ensure Internet is enabled in Notebook Settings."
    ) from last_error


def is_within_directory(directory, target):
    directory = Path(directory).resolve()
    target = Path(target).resolve()
    return target == directory or str(target).startswith(str(directory) + os.sep)


def safe_extract_zip(zip_path, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = dest_dir / member.filename
            if not is_within_directory(dest_dir, target):
                raise RuntimeError(f"Unsafe zip member path: {member.filename}")
        zf.extractall(dest_dir)


def safe_extract_tar(tar_path, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path) as tf:
        for member in tf.getmembers():
            target = dest_dir / member.name
            if not is_within_directory(dest_dir, target):
                raise RuntimeError(f"Unsafe tar member path: {member.name}")
        tf.extractall(dest_dir)


def extract_archive(archive_path, dest_dir):
    archive_path = Path(archive_path)
    name = archive_path.name.lower()
    print(f"Extracting {archive_path.name} -> {dest_dir}")
    if name.endswith(".zip"):
        safe_extract_zip(archive_path, dest_dir)
    elif name.endswith((".tar", ".tar.gz", ".tgz", ".tar.bz2", ".tbz2")):
        safe_extract_tar(archive_path, dest_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")


def find_dir(root, dirname):
    root = Path(root)
    for p in root.rglob(dirname):
        if p.is_dir():
            return p
    return None


def has_images(path):
    path = Path(path)
    return path.exists() and any(p.suffix.lower() in IMG_EXTS for p in path.rglob("*"))


def prepare_caltech101():
    existing = find_dir(DATA_DIR, "101_ObjectCategories")
    if existing is not None and has_images(existing):
        print("Caltech101 already extracted:", existing)
        return existing

    info = ARCHIVES["caltech101"]
    archive = download_file(info["url"], info["path"], info["md5"])
    raw_dir = DATA_DIR / "caltech101"
    raw_dir.mkdir(parents=True, exist_ok=True)

    if find_dir(raw_dir, "101_ObjectCategories") is None:
        extract_archive(archive, raw_dir)

    image_root = find_dir(raw_dir, "101_ObjectCategories")
    if image_root is None:
        nested_archives = [
            p for p in raw_dir.rglob("*")
            if p.is_file() and p.name.lower() in {
                "101_objectcategories.tar.gz",
                "101_objectcategories.tgz",
                "101_objectcategories.tar",
            }
        ]
        for nested in nested_archives:
            extract_archive(nested, raw_dir)
            image_root = find_dir(raw_dir, "101_ObjectCategories")
            if image_root is not None:
                break

    if image_root is None or not has_images(image_root):
        raise RuntimeError("Could not locate Caltech101 101_ObjectCategories after extraction.")
    print("Caltech101 image root:", image_root)
    return image_root


def prepare_cub2002011():
    existing = find_dir(DATA_DIR, "CUB_200_2011")
    if existing is not None and (existing / "images.txt").exists() and (existing / "images").exists():
        print("CUB-200-2011 already extracted:", existing)
        return existing

    info = ARCHIVES["cub2002011"]
    archive = download_file(info["url"], info["path"], info["md5"])
    raw_dir = DATA_DIR / "cub2002011"
    raw_dir.mkdir(parents=True, exist_ok=True)

    if find_dir(raw_dir, "CUB_200_2011") is None:
        extract_archive(archive, raw_dir)

    cub_root = find_dir(raw_dir, "CUB_200_2011")
    if cub_root is None or not (cub_root / "images.txt").exists():
        cub_root = find_dir(DATA_DIR, "CUB_200_2011")
    if cub_root is None or not (cub_root / "images.txt").exists():
        raise RuntimeError("Could not locate CUB_200_2011 metadata after extraction.")
    print("CUB-200-2011 root:", cub_root)
    return cub_root


caltech_root = prepare_caltech101()
cub_root = prepare_cub2002011()

## 5. Load metadata

Metadata chuẩn hóa gồm:
- `path`, `class_id`, `class_name`, `split`
- `width`, `height`, `aspect_ratio`
- CUB có thêm `bbox_x`, `bbox_y`, `bbox_w`, `bbox_h`, `bbox_area_ratio`

In [ ]:
def read_image_size(path):
    try:
        with Image.open(path) as img:
            return img.size
    except Exception as exc:
        print(f"Could not read image size: {path} | {exc}")
        return np.nan, np.nan


def add_image_sizes(df, path_col="path", desc="images"):
    widths, heights = [], []
    for p in tqdm(df[path_col].tolist(), desc=f"Reading sizes: {desc}"):
        w, h = read_image_size(p)
        widths.append(w)
        heights.append(h)
    out = df.copy()
    out["width"] = widths
    out["height"] = heights
    out["aspect_ratio"] = out["width"] / out["height"]
    return out


def load_caltech101_metadata(image_root):
    image_root = Path(image_root)
    class_dirs = sorted([p for p in image_root.iterdir() if p.is_dir()], key=lambda p: p.name.lower())
    class_to_id = {p.name: i + 1 for i, p in enumerate(class_dirs)}

    records = []
    for class_dir in class_dirs:
        image_paths = sorted([p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS])
        for p in image_paths:
            records.append({
                "dataset": "caltech101",
                "path": str(p),
                "rel_path": str(p.relative_to(image_root)),
                "class_id": class_to_id[class_dir.name],
                "class_name": class_dir.name,
                "split": "all",
            })

    df = pd.DataFrame(records).sort_values(["class_id", "path"]).reset_index(drop=True)
    df = add_image_sizes(df, desc="Caltech101")
    df["emb_index"] = np.arange(len(df))
    return df


def load_cub_metadata(root):
    root = Path(root)
    images = pd.read_csv(root / "images.txt", sep=r"\s+", names=["image_id", "rel_path"])
    labels = pd.read_csv(root / "image_class_labels.txt", sep=r"\s+", names=["image_id", "class_id"])
    split = pd.read_csv(root / "train_test_split.txt", sep=r"\s+", names=["image_id", "is_train"])
    classes = pd.read_csv(root / "classes.txt", sep=r"\s+", names=["class_id", "class_name_raw"])
    bbox = pd.read_csv(root / "bounding_boxes.txt", sep=r"\s+", names=["image_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h"])

    classes["class_name"] = (
        classes["class_name_raw"]
        .str.replace(r"^\d+\.", "", regex=True)
        .str.replace("_", " ", regex=False)
    )

    df = images.merge(labels, on="image_id", how="left")
    df = df.merge(split, on="image_id", how="left")
    df = df.merge(classes[["class_id", "class_name", "class_name_raw"]], on="class_id", how="left")
    df = df.merge(bbox, on="image_id", how="left")
    df["dataset"] = "cub2002011"
    df["split"] = np.where(df["is_train"].astype(int) == 1, "train", "test")
    df["path"] = df["rel_path"].apply(lambda x: str(root / "images" / x))

    missing = df[~df["path"].apply(lambda p: Path(p).exists())]
    if len(missing) > 0:
        raise RuntimeError(f"Missing {len(missing)} CUB images. First missing: {missing.iloc[0]['path']}")

    df = df[[
        "dataset", "image_id", "path", "rel_path", "class_id", "class_name", "class_name_raw",
        "split", "is_train", "bbox_x", "bbox_y", "bbox_w", "bbox_h",
    ]].sort_values("image_id").reset_index(drop=True)
    df = add_image_sizes(df, desc="CUB-200-2011")
    image_area = df["width"] * df["height"]
    bbox_area = df["bbox_w"].clip(lower=0) * df["bbox_h"].clip(lower=0)
    df["bbox_area_ratio"] = (bbox_area / image_area).clip(lower=0, upper=1)
    df["emb_index"] = np.arange(len(df))
    return df


caltech_df = load_caltech101_metadata(caltech_root)
cub_df = load_cub_metadata(cub_root)

print("Caltech101:", caltech_df.shape, "classes:", caltech_df["class_id"].nunique())
print("CUB-200-2011:", cub_df.shape, "classes:", cub_df["class_id"].nunique())
print("CUB split counts:")
print(cub_df["split"].value_counts())

caltech_df.to_csv(REPORT_DIR / "metadata_caltech101.csv", index=False)
cub_df.to_csv(REPORT_DIR / "metadata_cub2002011.csv", index=False)
print("Saved metadata CSV files to", REPORT_DIR)

## 6. EDA metadata và CSV reports

In [ ]:
def dataset_overview(df, dataset_name):
    row = {
        "dataset": dataset_name,
        "n_images": len(df),
        "n_classes": int(df["class_id"].nunique()),
        "split_counts": "; ".join([f"{k}:{v}" for k, v in df["split"].value_counts().to_dict().items()]),
        "width_min": float(df["width"].min()),
        "width_median": float(df["width"].median()),
        "width_mean": float(df["width"].mean()),
        "width_max": float(df["width"].max()),
        "height_min": float(df["height"].min()),
        "height_median": float(df["height"].median()),
        "height_mean": float(df["height"].mean()),
        "height_max": float(df["height"].max()),
        "aspect_ratio_min": float(df["aspect_ratio"].min()),
        "aspect_ratio_median": float(df["aspect_ratio"].median()),
        "aspect_ratio_mean": float(df["aspect_ratio"].mean()),
        "aspect_ratio_max": float(df["aspect_ratio"].max()),
    }
    if "bbox_area_ratio" in df.columns:
        row.update({
            "bbox_area_ratio_min": float(df["bbox_area_ratio"].min()),
            "bbox_area_ratio_median": float(df["bbox_area_ratio"].median()),
            "bbox_area_ratio_mean": float(df["bbox_area_ratio"].mean()),
            "bbox_area_ratio_max": float(df["bbox_area_ratio"].max()),
            "bbox_area_ratio_lt_025": float((df["bbox_area_ratio"] < 0.25).mean()),
            "bbox_area_ratio_lt_050": float((df["bbox_area_ratio"] < 0.50).mean()),
        })
    return row


overview_df = pd.DataFrame([
    dataset_overview(caltech_df, "caltech101"),
    dataset_overview(cub_df, "cub2002011"),
])
overview_df.to_csv(REPORT_DIR / "dataset_overview.csv", index=False)

combined_df = pd.concat([caltech_df, cub_df], ignore_index=True, sort=False)

class_counts = (
    combined_df.groupby(["dataset", "class_id", "class_name"])
    .agg(n_images=("path", "count"))
    .reset_index()
)
split_counts = (
    combined_df.groupby(["dataset", "class_id", "class_name", "split"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
class_counts = class_counts.merge(split_counts, on=["dataset", "class_id", "class_name"], how="left")
class_counts.to_csv(REPORT_DIR / "class_counts.csv", index=False)

size_stats = (
    combined_df.groupby("dataset")[["width", "height", "aspect_ratio"]]
    .describe()
    .reset_index()
)
size_stats.to_csv(REPORT_DIR / "image_size_stats.csv", index=False)

if "bbox_area_ratio" in cub_df.columns:
    cub_bbox_stats = cub_df["bbox_area_ratio"].describe().to_frame(name="bbox_area_ratio")
    cub_bbox_stats.to_csv(REPORT_DIR / "cub_bbox_area_ratio_stats.csv")

print("Saved reports:")
for p in sorted(REPORT_DIR.glob("*.csv")):
    print("-", p)

display(overview_df)
display(class_counts.groupby("dataset")["n_images"].describe())

## 7. Visualization EDA

In [ ]:
def save_and_show(fig, filename, dpi=160):
    out_path = FIG_DIR / filename
    fig.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    print("Saved figure:", out_path)
    if SHOW_FIGURES:
        display(fig)
    plt.close(fig)


def short_label(text, max_len=35):
    text = str(text)
    return text if len(text) <= max_len else text[: max_len - 3] + "..."


def plot_class_count_hist(df, dataset_name):
    counts = df.groupby("class_name").size().values
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(counts, bins=30, color="#3b82f6", edgecolor="white")
    ax.set_title(f"{dataset_name}: histogram of images per class")
    ax.set_xlabel("Images per class")
    ax.set_ylabel("Number of classes")
    ax.grid(alpha=0.25)
    save_and_show(fig, f"hist_images_per_class_{dataset_name}.png")


def plot_width_height_scatter(df, dataset_name, max_points=5000):
    sample = df.sample(min(len(df), max_points), random_state=SEED)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(sample["width"], sample["height"], s=8, alpha=0.35, color="#0f766e")
    ax.set_title(f"{dataset_name}: width vs height")
    ax.set_xlabel("Width")
    ax.set_ylabel("Height")
    ax.grid(alpha=0.25)
    save_and_show(fig, f"scatter_width_height_{dataset_name}.png")


def plot_aspect_ratio_hist(df, dataset_name):
    fig, ax = plt.subplots(figsize=(8, 4))
    values = df["aspect_ratio"].replace([np.inf, -np.inf], np.nan).dropna()
    ax.hist(values, bins=50, color="#f59e0b", edgecolor="white")
    ax.axvline(values.median(), color="black", linestyle="--", linewidth=1.2, label=f"median={values.median():.2f}")
    ax.set_title(f"{dataset_name}: aspect ratio histogram")
    ax.set_xlabel("width / height")
    ax.set_ylabel("Number of images")
    ax.legend()
    ax.grid(alpha=0.25)
    save_and_show(fig, f"hist_aspect_ratio_{dataset_name}.png")


def plot_random_image_grid(df, dataset_name, filename, n=16, cols=4):
    sample = df.sample(min(len(df), n), random_state=SEED)
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.0, rows * 3.2))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, (_, row) in zip(axes, sample.iterrows()):
        with Image.open(row["path"]) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(short_label(row["class_name"], 28), fontsize=8)
        ax.axis("off")
    fig.suptitle(f"Random images: {dataset_name}", y=1.02, fontsize=13)
    save_and_show(fig, filename)


def crop_from_bbox(image, row):
    x1 = max(0, int(round(row["bbox_x"])))
    y1 = max(0, int(round(row["bbox_y"])))
    x2 = min(image.width, int(round(row["bbox_x"] + row["bbox_w"])))
    y2 = min(image.height, int(round(row["bbox_y"] + row["bbox_h"])))
    if x2 <= x1 or y2 <= y1:
        return image.copy()
    return image.crop((x1, y1, x2, y2))


def plot_cub_full_vs_bbox(df, n=8):
    sample = df.dropna(subset=["bbox_x", "bbox_y", "bbox_w", "bbox_h"]).sample(min(n, len(df)), random_state=SEED)
    fig, axes = plt.subplots(len(sample), 2, figsize=(8, len(sample) * 2.4))
    if len(sample) == 1:
        axes = np.array([axes])
    for r, (_, row) in enumerate(sample.iterrows()):
        with Image.open(row["path"]) as img:
            img = img.convert("RGB")
            boxed = img.copy()
            draw = ImageDraw.Draw(boxed)
            x1 = max(0, int(round(row["bbox_x"])))
            y1 = max(0, int(round(row["bbox_y"])))
            x2 = min(img.width, int(round(row["bbox_x"] + row["bbox_w"])))
            y2 = min(img.height, int(round(row["bbox_y"] + row["bbox_h"])))
            draw.rectangle((x1, y1, x2, y2), outline="red", width=max(2, img.width // 150))
            crop = crop_from_bbox(img, row)
        axes[r, 0].imshow(boxed)
        axes[r, 0].set_title(short_label(row["class_name"], 32), fontsize=8)
        axes[r, 0].axis("off")
        axes[r, 1].imshow(crop)
        axes[r, 1].set_title(f"bbox crop | area ratio={row['bbox_area_ratio']:.2f}", fontsize=8)
        axes[r, 1].axis("off")
    fig.suptitle("CUB full image with bbox vs bbox crop", y=1.01, fontsize=13)
    save_and_show(fig, "cub_full_image_vs_bbox_crop.png")


for name, df in [("caltech101", caltech_df), ("cub2002011", cub_df)]:
    plot_class_count_hist(df, name)
    plot_width_height_scatter(df, name)
    plot_aspect_ratio_hist(df, name)

plot_random_image_grid(caltech_df, "Caltech101", "random_grid_caltech101.png")
plot_random_image_grid(cub_df, "CUB-200-2011", "random_grid_cub2002011.png")
plot_cub_full_vs_bbox(cub_df)

## 8. Extract CLIP embeddings

Baseline dùng `open_clip_torch`, model mặc định `ViT-B-32` pretrained `openai`. Embedding được normalize L2 và cache vào `/kaggle/working/eda_outputs/cache`.

In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = [str(p) for p in paths]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        with Image.open(path) as img:
            image = img.convert("RGB")
        return self.transform(image), idx


def cache_name(dataset_name):
    safe_model = MODEL_NAME.replace("/", "-")
    safe_pretrained = PRETRAINED.replace("/", "-")
    return CACHE_DIR / f"clip_{safe_model}_{safe_pretrained}_{dataset_name}.npz"


def load_embedding_cache(path, paths):
    path = Path(path)
    if not path.exists():
        return None
    expected_paths = [str(p) for p in paths]
    try:
        with np.load(path, allow_pickle=False) as data:
            cached_paths = data["paths"].astype(str).tolist()
            if cached_paths == expected_paths:
                emb = data["embeddings"].astype("float32")
                print(f"Loaded cached embeddings: {path} | shape={emb.shape}")
                return emb
            print(f"Cache path list mismatch, recomputing: {path}")
    except Exception as exc:
        print(f"Could not read cache {path}: {exc}. Recomputing.")
    return None


def save_embedding_cache(path, embeddings, paths):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        embeddings=embeddings.astype("float32"),
        paths=np.array([str(p) for p in paths], dtype=str),
        model=np.array(MODEL_NAME),
        pretrained=np.array(PRETRAINED),
    )
    print(f"Saved embeddings cache: {path} | shape={embeddings.shape}")


def extract_clip_embeddings(df, dataset_name, model, preprocess):
    paths = df["path"].astype(str).tolist()
    out_cache = cache_name(dataset_name)
    cached = load_embedding_cache(out_cache, paths)
    if cached is not None:
        return cached

    ds = ImagePathDataset(paths, preprocess)
    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
    )

    embeddings = np.zeros((len(ds), model.visual.output_dim), dtype="float32")
    model.eval()
    with torch.no_grad():
        for images, idx in tqdm(loader, desc=f"CLIP embeddings: {dataset_name}"):
            images = images.to(DEVICE, non_blocking=True)
            feats = model.encode_image(images)
            feats = feats / feats.norm(dim=-1, keepdim=True).clamp_min(1e-12)
            embeddings[idx.numpy()] = feats.detach().cpu().float().numpy()

    save_embedding_cache(out_cache, embeddings, paths)
    return embeddings


print(f"Loading CLIP model: {MODEL_NAME}, pretrained={PRETRAINED}")
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
model = model.to(DEVICE).eval()

caltech_emb = extract_clip_embeddings(caltech_df, "caltech101", model, preprocess)
cub_emb = extract_clip_embeddings(cub_df, "cub2002011", model, preprocess)

print("Caltech embeddings:", caltech_emb.shape)
print("CUB embeddings:", cub_emb.shape)

## 9. Retrieval baseline

Protocol:
- Caltech101: toàn bộ ảnh là query và gallery, exclude self-match.
- CUB-200-2011: query là test split, gallery là train + test, exclude self-match nếu query có trong gallery.
- Positive là ảnh cùng class.

Để tránh giữ full similarity matrix NxN trong bộ nhớ, evaluation chạy theo batch query và chỉ giữ một block similarity tại mỗi thời điểm.

In [ ]:
def average_precision_from_order(order, gallery_labels, query_label, self_pos=None):
    pos = gallery_labels[order] == query_label
    if self_pos is not None:
        pos[order == self_pos] = False
    total_pos = int((gallery_labels == query_label).sum())
    if self_pos is not None and gallery_labels[self_pos] == query_label:
        total_pos -= 1
    if total_pos <= 0:
        return np.nan
    hit_ranks = np.flatnonzero(pos) + 1
    if len(hit_ranks) == 0:
        return 0.0
    precisions = np.arange(1, len(hit_ranks) + 1, dtype=np.float64) / hit_ranks
    return float(precisions.sum() / total_pos)


def evaluate_retrieval(dataset_name, embeddings, df, query_indices, gallery_indices, topk=TOPK, chunk_size=SIM_CHUNK_SIZE):
    query_df = df.iloc[query_indices].reset_index(drop=True).copy()
    gallery_df = df.iloc[gallery_indices].reset_index(drop=True).copy()
    query_emb = embeddings[query_indices].astype("float32")
    gallery_emb = embeddings[gallery_indices].astype("float32")

    gallery_labels = gallery_df["class_id"].to_numpy()
    gallery_paths = gallery_df["path"].astype(str).tolist()
    gallery_path_to_pos = {p: i for i, p in enumerate(gallery_paths)}

    gallery_t = torch.from_numpy(gallery_emb).to(DEVICE)
    rows = []

    for start in tqdm(range(0, len(query_df), chunk_size), desc=f"Retrieval eval: {dataset_name}"):
        end = min(start + chunk_size, len(query_df))
        q_t = torch.from_numpy(query_emb[start:end]).to(DEVICE)
        sims = q_t @ gallery_t.T

        batch_self_pos = []
        for local_i, (_, qrow) in enumerate(query_df.iloc[start:end].iterrows()):
            self_pos = gallery_path_to_pos.get(str(qrow["path"]))
            batch_self_pos.append(self_pos)
            if self_pos is not None:
                sims[local_i, self_pos] = -float("inf")

        order_t = torch.argsort(sims, dim=1, descending=True)
        order_np = order_t.detach().cpu().numpy()
        sims_np = sims.detach().cpu().numpy()

        for local_i, (_, qrow) in enumerate(query_df.iloc[start:end].iterrows()):
            q_label = int(qrow["class_id"])
            order = order_np[local_i]
            self_pos = batch_self_pos[local_i]
            if self_pos is not None:
                order_no_self = order[order != self_pos]
            else:
                order_no_self = order
            top_indices = order_no_self[:topk]
            top_labels = gallery_labels[top_indices]
            top_scores = sims_np[local_i, top_indices]
            top_rows = gallery_df.iloc[top_indices]

            recalls = {f"recall@{k}": float(np.any(top_labels[:k] == q_label)) for k in [1, 5, 10]}
            ap = average_precision_from_order(order, gallery_labels, q_label, self_pos=self_pos)

            top_paths = top_rows["path"].astype(str).tolist()
            top_class_ids = top_rows["class_id"].astype(int).tolist()
            top_class_names = top_rows["class_name"].astype(str).tolist()
            top_scores_list = [float(x) for x in top_scores.tolist()]

            rows.append({
                "dataset": dataset_name,
                "query_pos": int(start + local_i),
                "query_path": str(qrow["path"]),
                "query_class_id": q_label,
                "query_class_name": str(qrow["class_name"]),
                "query_split": str(qrow["split"]),
                "top1_path": top_paths[0] if top_paths else None,
                "top1_class_id": top_class_ids[0] if top_class_ids else None,
                "top1_class_name": top_class_names[0] if top_class_names else None,
                "top1_score": top_scores_list[0] if top_scores_list else np.nan,
                "topk_paths": "|".join(top_paths),
                "topk_class_ids": "|".join(map(str, top_class_ids)),
                "topk_class_names": "|".join(top_class_names),
                "topk_scores": "|".join([f"{s:.6f}" for s in top_scores_list]),
                "ap": ap,
                **recalls,
            })

        del sims, order_t, q_t
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    results = pd.DataFrame(rows)
    metrics = {
        "dataset": dataset_name,
        "n_queries": len(query_df),
        "n_gallery": len(gallery_df),
        "recall_at_1": float(results["recall@1"].mean()),
        "recall_at_5": float(results["recall@5"].mean()),
        "recall_at_10": float(results["recall@10"].mean()),
        "mAP": float(results["ap"].mean()),
        "macro_recall_at_1": float(results.groupby("query_class_id")["recall@1"].mean().mean()),
    }
    return results, metrics


caltech_query_idx = np.arange(len(caltech_df))
caltech_gallery_idx = np.arange(len(caltech_df))

cub_query_idx = cub_df.index[cub_df["split"] == "test"].to_numpy()
cub_gallery_idx = np.arange(len(cub_df))

caltech_results, caltech_metrics = evaluate_retrieval(
    "caltech101", caltech_emb, caltech_df, caltech_query_idx, caltech_gallery_idx
)
cub_results, cub_metrics = evaluate_retrieval(
    "cub2002011", cub_emb, cub_df, cub_query_idx, cub_gallery_idx
)

retrieval_results = pd.concat([caltech_results, cub_results], ignore_index=True)
retrieval_metrics = pd.DataFrame([caltech_metrics, cub_metrics])

retrieval_results.to_csv(REPORT_DIR / "retrieval_baseline_results.csv", index=False)
retrieval_metrics.to_csv(REPORT_DIR / "retrieval_baseline_metrics.csv", index=False)

print("Retrieval metrics:")
display(retrieval_metrics)
print("Saved retrieval reports to", REPORT_DIR)

## 10. Similarity separation, confusion pairs và hard queries

In [ ]:
def sample_pair_similarities(embeddings, labels, n_pairs=PAIR_SIM_SAMPLES, seed=SEED):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    unique_labels = np.unique(labels)
    groups = {label: np.flatnonzero(labels == label) for label in unique_labels}
    same_classes = [label for label, idx in groups.items() if len(idx) >= 2]

    n_same = min(n_pairs, sum(len(idx) * (len(idx) - 1) // 2 for idx in groups.values() if len(idx) >= 2))
    same = np.empty(n_same, dtype="float32")
    for i in range(n_same):
        c = rng.choice(same_classes)
        a, b = rng.choice(groups[c], size=2, replace=False)
        same[i] = float(np.dot(embeddings[a], embeddings[b]))

    diff = np.empty(n_pairs, dtype="float32")
    for i in range(n_pairs):
        c1, c2 = rng.choice(unique_labels, size=2, replace=False)
        a = rng.choice(groups[c1])
        b = rng.choice(groups[c2])
        diff[i] = float(np.dot(embeddings[a], embeddings[b]))

    return same, diff


def plot_similarity_hist(dataset_name, same, diff):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(diff, bins=60, alpha=0.55, density=True, label="different class", color="#64748b")
    ax.hist(same, bins=60, alpha=0.55, density=True, label="same class", color="#16a34a")
    ax.axvline(np.median(diff), color="#334155", linestyle="--", linewidth=1)
    ax.axvline(np.median(same), color="#166534", linestyle="--", linewidth=1)
    ax.set_title(f"{dataset_name}: cosine similarity distribution")
    ax.set_xlabel("cosine similarity")
    ax.set_ylabel("density")
    ax.legend()
    ax.grid(alpha=0.25)
    save_and_show(fig, f"hist_similarity_same_vs_diff_{dataset_name}.png")


sim_summary_rows = []
for dataset_name, emb, df in [
    ("caltech101", caltech_emb, caltech_df),
    ("cub2002011", cub_emb, cub_df),
]:
    same, diff = sample_pair_similarities(emb, df["class_id"].to_numpy(), n_pairs=PAIR_SIM_SAMPLES)
    plot_similarity_hist(dataset_name, same, diff)
    sim_summary_rows.append({
        "dataset": dataset_name,
        "same_mean": float(np.mean(same)),
        "same_median": float(np.median(same)),
        "same_p10": float(np.percentile(same, 10)),
        "same_p90": float(np.percentile(same, 90)),
        "diff_mean": float(np.mean(diff)),
        "diff_median": float(np.median(diff)),
        "diff_p10": float(np.percentile(diff, 10)),
        "diff_p90": float(np.percentile(diff, 90)),
        "median_gap": float(np.median(same) - np.median(diff)),
    })

similarity_summary = pd.DataFrame(sim_summary_rows)
similarity_summary.to_csv(REPORT_DIR / "similarity_same_vs_diff_summary.csv", index=False)
display(similarity_summary)


def build_confusion_pairs(results, topn=50):
    wrong = results[results["recall@1"] == 0].copy()
    if len(wrong) == 0:
        return pd.DataFrame(columns=[
            "dataset", "query_class_id", "query_class_name", "top1_class_id", "top1_class_name", "n_errors",
            "mean_top1_score", "mean_ap",
        ])
    out = (
        wrong.groupby(["dataset", "query_class_id", "query_class_name", "top1_class_id", "top1_class_name"])
        .agg(n_errors=("query_path", "count"), mean_top1_score=("top1_score", "mean"), mean_ap=("ap", "mean"))
        .reset_index()
        .sort_values(["dataset", "n_errors", "mean_top1_score"], ascending=[True, False, False])
    )
    return out.groupby("dataset", group_keys=False).head(topn).reset_index(drop=True)


def build_hard_queries(results, topn=100):
    hard = results.copy()
    hard["hard_score"] = (1 - hard["recall@10"]) * 2 + (1 - hard["recall@1"]) + (1 - hard["ap"].fillna(0))
    hard = hard.sort_values(["dataset", "hard_score", "ap", "top1_score"], ascending=[True, False, True, False])
    return hard.groupby("dataset", group_keys=False).head(topn).reset_index(drop=True)


confused_pairs = build_confusion_pairs(retrieval_results)
hard_queries = build_hard_queries(retrieval_results)

confused_pairs.to_csv(REPORT_DIR / "top_confused_class_pairs.csv", index=False)
hard_queries.to_csv(REPORT_DIR / "hard_queries.csv", index=False)

print("Top confused class pairs:")
display(confused_pairs.head(20))
print("Hard queries:")
display(hard_queries.head(20))

## 11. Visualization retrieval baseline

In [ ]:
def plot_metrics_bar(metrics_df):
    metric_cols = ["recall_at_1", "recall_at_5", "recall_at_10", "mAP"]
    datasets = metrics_df["dataset"].tolist()
    x = np.arange(len(metric_cols))
    width = 0.36
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for i, dataset in enumerate(datasets):
        values = metrics_df.loc[metrics_df["dataset"] == dataset, metric_cols].iloc[0].values.astype(float)
        ax.bar(x + (i - 0.5) * width, values, width=width, label=dataset)
        for xi, v in zip(x + (i - 0.5) * width, values):
            ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(["Recall@1", "Recall@5", "Recall@10", "mAP"])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("CLIP retrieval baseline metrics")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    save_and_show(fig, "bar_retrieval_metrics_clip_baseline.png")


def plot_confused_pairs(confused_df, topn=12):
    if len(confused_df) == 0:
        print("No top-1 errors found. Skip confused-pair plot.")
        return
    datasets = confused_df["dataset"].unique().tolist()
    fig, axes = plt.subplots(1, len(datasets), figsize=(7 * len(datasets), 5), squeeze=False)
    for ax, dataset in zip(axes[0], datasets):
        sub = confused_df[confused_df["dataset"] == dataset].head(topn).iloc[::-1]
        labels = [f"{short_label(a, 18)} -> {short_label(b, 18)}" for a, b in zip(sub["query_class_name"], sub["top1_class_name"])]
        ax.barh(labels, sub["n_errors"], color="#ef4444")
        ax.set_title(f"{dataset}: top confused class pairs")
        ax.set_xlabel("Top-1 error count")
        ax.grid(axis="x", alpha=0.25)
    save_and_show(fig, "bar_top_confused_class_pairs.png")


def parse_pipe_list(value):
    if pd.isna(value) or value is None:
        return []
    return str(value).split("|") if str(value) else []


def plot_retrieval_examples(results, dataset_name, filename, n_examples=6, k=5):
    sub = results[results["dataset"] == dataset_name].copy()
    wrong = sub[sub["recall@1"] == 0].sort_values("ap").head(n_examples // 2)
    correct = sub[sub["recall@1"] == 1].sort_values("ap", ascending=False).head(n_examples - len(wrong))
    examples = pd.concat([wrong, correct], ignore_index=True)
    if len(examples) == 0:
        print(f"No examples for {dataset_name}")
        return

    rows = len(examples)
    cols = k + 1
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.5))
    if rows == 1:
        axes = np.array([axes])

    for r, (_, row) in enumerate(examples.iterrows()):
        paths = [row["query_path"]] + parse_pipe_list(row["topk_paths"])[:k]
        class_names = [row["query_class_name"]] + parse_pipe_list(row["topk_class_names"])[:k]
        scores = [None] + parse_pipe_list(row["topk_scores"])[:k]
        query_class = row["query_class_name"]
        for c in range(cols):
            ax = axes[r, c]
            ax.axis("off")
            if c >= len(paths):
                continue
            with Image.open(paths[c]) as img:
                ax.imshow(img.convert("RGB"))
            if c == 0:
                title = f"Q\n{short_label(query_class, 20)}"
                color = "black"
            else:
                is_correct = class_names[c] == query_class
                title = f"#{c} {scores[c][:5] if scores[c] else ''}\n{short_label(class_names[c], 20)}"
                color = "green" if is_correct else "red"
                for spine in ax.spines.values():
                    spine.set_visible(True)
                    spine.set_linewidth(2.5)
                    spine.set_edgecolor(color)
            ax.set_title(title, fontsize=8, color=color)
    fig.suptitle(f"Retrieval examples: {dataset_name}", y=1.01, fontsize=13)
    save_and_show(fig, filename)


plot_metrics_bar(retrieval_metrics)
plot_confused_pairs(confused_pairs)
plot_retrieval_examples(retrieval_results, "caltech101", "retrieval_examples_caltech101.png")
plot_retrieval_examples(retrieval_results, "cub2002011", "retrieval_examples_cub2002011.png")

## 12. Error analysis nhanh

In [ ]:
def print_dataset_diagnostics(dataset_name, metrics_df, sim_summary_df, confused_df):
    m = metrics_df[metrics_df["dataset"] == dataset_name].iloc[0]
    s = sim_summary_df[sim_summary_df["dataset"] == dataset_name].iloc[0]
    print(f"\n=== {dataset_name} ===")
    print(f"Recall@1={m['recall_at_1']:.3f} | Recall@5={m['recall_at_5']:.3f} | Recall@10={m['recall_at_10']:.3f} | mAP={m['mAP']:.3f} | macro Recall@1={m['macro_recall_at_1']:.3f}")
    print(f"Same-vs-different median cosine gap: {s['median_gap']:.3f} (same={s['same_median']:.3f}, diff={s['diff_median']:.3f})")
    top = confused_df[confused_df["dataset"] == dataset_name].head(5)
    if len(top) > 0:
        print("Top confused class pairs:")
        for _, row in top.iterrows():
            print(f"- {row['query_class_name']} -> {row['top1_class_name']}: {int(row['n_errors'])} errors")
    else:
        print("No top-1 confusion pairs.")


print_dataset_diagnostics("caltech101", retrieval_metrics, similarity_summary, confused_pairs)
print_dataset_diagnostics("cub2002011", retrieval_metrics, similarity_summary, confused_pairs)

bbox_median = cub_df["bbox_area_ratio"].median()
bbox_lt025 = (cub_df["bbox_area_ratio"] < 0.25).mean()
bbox_lt050 = (cub_df["bbox_area_ratio"] < 0.50).mean()
print("\n=== CUB bbox quick signal ===")
print(f"bbox area ratio median={bbox_median:.3f} | fraction <0.25={bbox_lt025:.3f} | fraction <0.50={bbox_lt050:.3f}")
if bbox_median < 0.35 or bbox_lt025 > 0.25:
    print("Many CUB objects occupy a relatively small image area. Object-centric preprocessing or bbox crop is worth testing.")
else:
    print("CUB objects often occupy a moderate or large image area. Bbox crop may still help, but full-image CLIP is a reasonable first baseline.")

## 13. Kết luận / khuyến nghị pipeline

Cell cuối in khuyến nghị ngắn dựa trên số liệu đã chạy.

In [ ]:
def metric_value(dataset, col):
    return float(retrieval_metrics.loc[retrieval_metrics["dataset"] == dataset, col].iloc[0])

cal_r1 = metric_value("caltech101", "recall_at_1")
cal_r5 = metric_value("caltech101", "recall_at_5")
cal_map = metric_value("caltech101", "mAP")
cub_r1 = metric_value("cub2002011", "recall_at_1")
cub_r5 = metric_value("cub2002011", "recall_at_5")
cub_map = metric_value("cub2002011", "mAP")

print("=== Short recommendations ===")

if cal_r1 >= 0.70 and cal_map >= 0.50:
    print(f"Caltech101: CLIP baseline looks strong enough for a simple retrieval pipeline (R@1={cal_r1:.3f}, R@5={cal_r5:.3f}, mAP={cal_map:.3f}).")
elif cal_r1 >= 0.50:
    print(f"Caltech101: simple CLIP retrieval is usable but not saturated (R@1={cal_r1:.3f}, R@5={cal_r5:.3f}, mAP={cal_map:.3f}). Reranking or stronger embeddings can help.")
else:
    print(f"Caltech101: CLIP baseline is weak for reliable retrieval (R@1={cal_r1:.3f}, R@5={cal_r5:.3f}, mAP={cal_map:.3f}). Consider stronger embeddings or preprocessing.")

if cub_r1 < cal_r1:
    print(f"CUB-200-2011: harder than Caltech101 under this protocol (CUB R@1={cub_r1:.3f} vs Caltech R@1={cal_r1:.3f}). Fine-grained bird classes make class-level retrieval more difficult.")
else:
    print(f"CUB-200-2011: not worse than Caltech101 by R@1 in this run (CUB R@1={cub_r1:.3f} vs Caltech R@1={cal_r1:.3f}), but inspect confusion pairs because CUB remains fine-grained.")

if bbox_median < 0.35 or bbox_lt025 > 0.25 or cub_r1 < 0.60:
    print(f"CUB bbox crop: worth testing. Median bbox/image area is {bbox_median:.3f}, and full-image CLIP R@1 is {cub_r1:.3f}.")
else:
    print(f"CUB bbox crop: optional follow-up. Median bbox/image area is {bbox_median:.3f}; full-image CLIP is a fair baseline.")

if min(cal_r1, cub_r1) < 0.70 or min(cal_map, cub_map) < 0.50:
    print("If CLIP is not good enough: try IsoCLIP, DINO/DINOv2 embeddings, object-centric preprocessing, query/gallery reranking, or class-aware hard negative analysis.")
else:
    print("CLIP baseline is reasonably strong. Next steps can focus on reranking, latency, memory, and robustness rather than replacing the encoder immediately.")

print("\nReports:", REPORT_DIR)
print("Figures:", FIG_DIR)
print("Embedding cache:", CACHE_DIR)